# Performance

## Measurement Conditions and Scope

The measurements describe the arXiv Fast Crawl/Recovery invocations reported
in Hieu's document and the Semantic Scholar reruns exported on **11 November
2026**. The documents identify Google Colab CPU-only as the execution environment.

Each chart identifies its stage, cache state, timer or storage scope. arXiv
invocation durations and Semantic Scholar rerun timings are presented separately.
Charts, tables and observations are generated at compilation from
`milestone1/data/reported_results.json`.

In [ ]:
from pathlib import Path
import sys

# Support execution from the project root or the notebook's own directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "myst.yml").is_file() and (p / "milestone1" / "reporting.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from milestone1.reporting import load_results, total, integer, percent, seconds, hms, table

data = load_results()
arxiv = data["arxiv"]["ranges"]
s2 = data["semantic_scholar"]["ranges"]

from milestone1.reporting import analysis, COLORS, chart_style, finish_figure
from matplotlib.ticker import FuncFormatter

plt = chart_style()
labels = [f"Range {r['range']}" for r in arxiv]
positions = list(range(len(labels)))
comma = FuncFormatter(lambda value, _: f"{value:,.0f}")

## How Much Time Is Spent on Fast Crawl and Recovery?

**Figure 1.** Recorded arXiv phase durations in decimal hours. Each horizontal
bar represents a range; phase values are labeled inside and their sum to the
right. Bar lengths use the printed phase times and labels are rounded to two
decimals.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.6), layout="constrained")
fast = [seconds(r["fast_hms"])/3600 for r in arxiv]
recovery = [seconds(r["recovery_hms"])/3600 for r in arxiv]
ax.barh(positions, fast, height=0.62, color=COLORS["blue"], label="Fast Crawl")
ax.barh(positions, recovery, left=fast, height=0.62,
        color=COLORS["orange"], label="Recovery")
for i, (f, r) in enumerate(zip(fast, recovery)):
    ax.text(f/2, i, f"{f:.2f}", ha="center", va="center",
            color="white", fontweight="bold", fontsize=13)
    ax.text(f+r/2, i, f"{r:.2f}", ha="center", va="center",
            color="#111827", fontweight="bold", fontsize=13)
    ax.text(f+r+0.3, i, f"{f+r:.2f} h", va="center",
            color="#111827", fontweight="bold", fontsize=12)
ax.set(yticks=positions, yticklabels=labels, xlim=(0, 22))
ax.set_xticks(range(0, 23, 2))
ax.invert_yaxis()
ax.set_xlabel("Recorded invocation time (hours)", fontsize=13, labelpad=10)
ax.set_title("arXiv: Fast Crawl and Recovery", fontsize=16, pad=16)
ax.tick_params(axis="both", labelsize=12)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0, pad=12)
ax.grid(axis="x", alpha=0.16)
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center",
           ncol=2, frameon=False, fontsize=13)
finish_figure(fig, "performance-arxiv-runtime",
              "Horizontal stacked bars show recorded Fast Crawl and Recovery hours for each range, with readable phase labels and totals.")

overall = data["arxiv"]["reported_overall"]
reported_time = seconds(overall["total_hms"])
assigned = total(arxiv, "assigned")
longest = max(arxiv, key=lambda r: seconds(r["total_hms"]))
shortest = min(arxiv, key=lambda r: seconds(r["total_hms"]))
analysis(
f"The document reports {overall['total_hms']} of invocation time "
    f"({reported_time/3600:.2f} hours), equivalent to {reported_time/assigned:.3f} recorded "
    f"seconds per assigned ID. Recovery contributes "
    f"{percent(seconds(overall['recovery_hms']), reported_time, 1)}. "
    f"Range totals run from {shortest['total_hms']} (Range {shortest['range']}) to "
    f"{longest['total_hms']} (Range {longest['range']}).",
    "Recovery accounts for a substantial share of the recorded arXiv processing. "
    "Reporting both phases captures the time spent on initial collection and subsequent retries.",
    "These durations describe the reported arXiv invocations. Phase bars use the printed "
    "per-range phase values; the overall figure above uses the document's reported total."
)

## What Do the Semantic Scholar Timers Include?

**Figure 2.** Fetch, crawl-loop and notebook-cell times from the cache-heavy
reruns. Each panel uses its own scale. Labels below each range give the number
of fresh publication lookups.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), layout="constrained")
fresh = [r["input_papers"]-r["cache_hits"]-r["invalid_id"] for r in s2]
tick_labels = [f"Range {r['range']}\n{n} fresh" for r,n in zip(s2,fresh)]
for ax, key, title, color in (
    (axes[0], "fetch_seconds", "Accumulated fresh-lookup fetch time", COLORS["blue"]),
    (axes[1], "crawl_seconds", "Crawl-loop wall time", COLORS["teal"]),
    (axes[2], "cell_seconds", "Notebook-cell wall time", COLORS["purple"]),
):
    values = [r[key] for r in s2]
    bars = ax.bar(positions, values, color=color)
    ax.bar_label(bars, labels=[f"{v:,.2f}" for v in values], padding=4, fontsize=9)
    ax.set(xticks=positions, xticklabels=tick_labels, ylim=(0, max(values)*1.2),
           ylabel="Time (seconds)", title=title)
    ax.grid(axis="y", alpha=0.18)
finish_figure(fig, "performance-semantic-scholar-timing",
              "Cache-heavy Semantic Scholar rerun timings with separate scales for fetch and cell time, and fresh lookup counts.")

cached_count = total(s2, "cache_hits")
inputs = total(s2, "input_papers")
fresh_count = inputs-cached_count-total(s2, "invalid_id")
largest_cell = max(s2, key=lambda r: r["cell_seconds"])
analysis(
f"Across the five reruns, {integer(cached_count)} / {integer(inputs)} inputs are cache hits "
    f"({percent(cached_count, inputs, 3)}), leaving {integer(fresh_count)} fresh lookups. "
    f"Their accumulated fetch time is {total(s2, 'fetch_seconds'):.4f} seconds, or "
    f"{total(s2, 'fetch_seconds')/fresh_count:.4f} seconds per fresh publication. "
    f"Range {largest_cell['range']} has the largest cell time ({largest_cell['cell_seconds']:.2f} seconds).",
    "The crawl loop includes cache reads, API requests, normalization and cache writes. "
    "The cell timer includes the surrounding cell work up to the metrics/CSV export boundary.",
    "Fresh fetch timers include pacing and retries for one publication and stop before "
    "normalization/cache writing. The measured publications were the 19 uncached inputs, "
    "all ending in not_indexed. Checkpoint loading and table export/validation lie outside the cell timer."
)

## What Memory Was Observed in the Reference Rerun?

**Figure 3.** Average and peak sampled Semantic Scholar notebook-process
RSS, converted from bytes to MiB. The two series share a zero-based scale;
each label shows the measured value to two decimal places.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.8), layout="constrained")
width = 0.32
averages = [r["average_rss_mib"] for r in s2]
peaks = [r["peak_rss_mib"] for r in s2]
avg_bars = ax.barh([i-width/2 for i in positions], averages, height=width,
                   color=COLORS["blue"], label="Average sampled RSS")
peak_bars = ax.barh([i+width/2 for i in positions], peaks, height=width,
                    color=COLORS["orange"], label="Peak sampled RSS")
for bars, values in ((avg_bars, averages), (peak_bars, peaks)):
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in values], padding=5, fontsize=10)
ax.set(yticks=positions, yticklabels=labels, xlim=(0, 165),
       xlabel="Notebook-process RSS (MiB)", title="Semantic Scholar: cache-heavy rerun memory")
ax.invert_yaxis()
ax.tick_params(labelsize=12)
ax.grid(axis="x", alpha=0.16)
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center", ncol=2, frameon=False)
finish_figure(fig, "performance-semantic-scholar-memory",
              "Horizontal bars compare average and peak notebook-process RSS on a shared zero-based MiB axis.")

analysis(
f"Sampled process peaks range from {min(peaks):.2f} to {max(peaks):.2f} MiB. "
    f"The largest within-range difference between peak and average RSS is "
    f"{max(b-a for a,b in zip(averages,peaks)):.2f} MiB.",
    "The five reruns show similar process-level memory footprints on the same zero-based scale.",
    "RSS is sampled from the current Python process every 0.5 seconds and includes the "
    "existing kernel/Colab baseline. The metrics contain the average, peak and sample count "
    "for each measured cell; child processes are outside this process-level measure."
)

## How Much Persistent Storage Do the arXiv Checkpoints Use?

**Figure 4.** Final compressed arXiv checkpoint sizes, including retained
source files and bookkeeping. Values are in MiB and describe persistent
checkpoint output.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.4), layout="constrained")
sizes = [r["checkpoint_mib"] for r in arxiv]
bars = ax.bar(positions, sizes, color=COLORS["teal"])
ax.bar_label(bars, labels=[f"{v:,.2f}" for v in sizes], padding=4)
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 350),
       ylabel="Compressed checkpoint storage (MiB)", title="arXiv: retained checkpoint storage")
ax.grid(axis="y", alpha=0.18)
finish_figure(fig, "performance-checkpoint-storage",
              "Final compressed arXiv checkpoint sizes range from 237.15 to 291.45 MiB.")

size = total(arxiv, "checkpoint_mib")
smallest = min(arxiv, key=lambda r: r["checkpoint_mib"])
largest = max(arxiv, key=lambda r: r["checkpoint_mib"])
analysis(
f"The retained arXiv checkpoints total {size:,.2f} MiB ({size/1024:.2f} GiB). "
    f"Range sizes run from {smallest['checkpoint_mib']:.2f} MiB (Range {smallest['range']}) "
    f"to {largest['checkpoint_mib']:.2f} MiB (Range {largest['range']}).",
    "Equal assigned ID counts produce different checkpoint volumes across the five ranges.",
    "This measurement is the compressed arXiv checkpoint footprint, including retained "
    "sources and bookkeeping. It describes the files saved by this stage."
)

## What Storage Was Measured for Semantic Scholar?

**Figure 5.** The left panel shows final S2 range-directory sizes after table
export, measured in Dat's document. The right shows the sampled peak of the
reference-cache directory during the crawl cell. Both use MiB and retain their
different directory and time scopes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained")
for ax, key, title, color in (
    (axes[0], "final_output_bytes", "Final S2 range directories", COLORS["teal"]),
    (axes[1], "peak_cache_disk_bytes", "Sampled peak: reference cache", COLORS["orange"]),
):
    values = [r[key]/1024**2 for r in s2]
    bars = ax.barh(positions, values, color=color)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in values], padding=4)
    ax.set(yticks=positions, yticklabels=labels, xlim=(0, 180),
           xlabel="Storage (MiB)", title=title)
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.18)
finish_figure(fig, "performance-semantic-scholar-storage",
              "Final uncompressed Semantic Scholar range directories and sampled cache-directory disk peaks.")

output_bytes = total(s2, "final_output_bytes")
analysis(
    f"Final S2 output totals {integer(output_bytes)} bytes ({output_bytes/1024**2:.2f} MiB). "
    f"Sampled cache-directory peaks range from {min(r['peak_cache_disk_bytes'] for r in s2)/1024**2:.2f} "
    f"to {max(r['peak_cache_disk_bytes'] for r in s2)/1024**2:.2f} MiB.",
    "The final directories retain per-publication reference JSON and the exported tables and "
    "reports. The sampled cache measurement describes the working reference files during collection.",
    "Final size sums files recursively under each S2 range directory after export. Cache disk "
    "sampling uses a 30-second interval and covers reference/ only. These are uncompressed "
    "JSON/CSV stage outputs; arXiv checkpoint archives have their own storage measure."
)

## Recorded Measurements by Run

**Table 1.** Run identifiers and measured stage results. RSS is converted
directly from the original byte values. Final S2 output size comes from the
post-export directory measurement.

In [ ]:
table(["Range", "Run ID", "Crawl (s)", "Cell (s)", "RSS peak (MiB)", "Final S2 output (bytes)"],
      [[f"Range {r['range']}", r["run_id"], f"{r['crawl_seconds']:.4f}",
        f"{r['cell_seconds']:.4f}", f"{r['peak_rss_mib']:.2f}", integer(r["final_output_bytes"])] for r in s2])

## HTTP Request Activity

**Table 2.** Publication lookups and HTTP attempts in each rerun. Retries
and HTTP 429 responses are counted at request level.

In [ ]:
table(["Range", "Fresh publications", "HTTP attempts", "Retries", "HTTP 429"],
      [[f"Range {r['range']}", str(r["input_papers"]-r["cache_hits"]),
        str(r["http_requests"]), str(r["http_retries"]), str(r["http_429"])] for r in s2])

The five reruns recorded 19 fresh publication lookups, 34 HTTP attempts,
15 retries and 15 HTTP 429 responses. Final publication outcomes are reported
separately on [Statistics](statistics.ipynb).